# B3 — equilibrium-augmented formulation: RECALIBRATED re-run (attempt 2)

**Follow-up paper, Professor Rabczuk's items 2 and 3.** Attempt 1 (`w_equilibrium=1.0`, full 50,000 iterations) **failed**: energy 181.8%, reaction 216.9%, region Cauchy field 1013.8% relative error against the held-out set, vs. the DEM baseline's 0.3%/3.0%.

**Real, measured root cause found since** (CPU-only, production model size, real sampled material/load batches, 3 seeds): the raw (unweighted) weak-form equilibrium residual is **6.5-10x LARGER** than the energy term at initialization (energy 2.6-5.2, residual 22-51, ratio 0.10-0.15) -- `w_equilibrium=1.0` was never actually "roughly balanced" as the original toy-model-only check suggested; at production scale it let the equilibrium term (which is also inherently noisy -- resampled with fresh, unseeded random virtual test fields every iteration, measured swinging ±40% between two evaluations of the identical displacement field) dominate the total loss, so training mostly chased a noisy, over-weighted target instead of the energy term that actually matters.

**The fix**: `w_equilibrium` corrected from 1.0 to **0.12** (the measured ratio, rounded). Verified on a CPU fixed-batch diagnostic (`diagnose_equilibrium_augmented_B3_scale.py --tiny --w_equilibrium 0.12`, 600 steps): `energy` settles to a stable plateau (~2.30) nearly identical to a pure-energy-only (`w_equilibrium=0`) reference run on the exact same batch (~2.25-2.30) -- the corrected weight no longer visibly perturbs the energy term's own healthy convergence, unlike `w=1.0`'s complete domination. `weak_form_residual` itself remains noisy run-to-run (same unseeded-resampling behavior as before) but this no longer destabilizes the term that actually drives learning.

**Honest caveat**: this fixes the one real, measured, quantified problem (weight miscalibration) -- the same kind of fix that worked for `train_B3.py`'s own `OUTPUT_SCALE` and mixed-stress's `P_OUTPUT_SCALE`/`w_equilibrium`. The fixed-batch CPU check is a necessary sanity gate, not a guarantee -- watch the real run's printed `energy` and `weak_form_residual` for the first few thousand iterations: `energy` should behave like the DEM baseline's own training (settling toward a low value, not diverging or oscillating wildly); if it does not, interrupt and report rather than letting the full 50,000 iterations run.

**Writes to a NEW output directory** (`b3_equilibrium_augmented_v2`), not attempt 1's own `b3_equilibrium_augmented`, so both attempts' checkpoints/logs stay separately inspectable. This checkpoint is still u-only, so the ORIGINAL `evaluate_B3_qois.py` applies unchanged for evaluation, same as attempt 1.

In [ ]:
# =====================================================================
#  CELL -- B3 equilibrium-augmented: attempt 2, RECALIBRATED w_equilibrium
#  (50,000 iterations, production model size), then evaluation with the
#  ORIGINAL evaluate_B3_qois.py (this checkpoint is u-only, same output
#  shape as the baseline). See the markdown cell above for why attempt 1
#  failed and what this attempt changes.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 -- if not, '
      'use Runtime > Restart session before trusting this run)')

print(f'\n{"=" * 90}')
print('=== Re-running the equilibrium-residual correctness check in this environment ===')
from omar_pfem.test_equilibrium_augmented_B3 import test_residual_tracks_energy_minimization
test_residual_tracks_energy_minimization()

from omar_pfem.train_B3_equilibrium_augmented import get_eq_args, train
from omar_pfem.data.data_generate_B3_dataset import DEFAULT_RESOLUTION

device = torch.device('cuda')
R = '/content/drive/MyDrive/pfem_run'
OUTPUT_DIR = f'{R}/b3_equilibrium_augmented_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

args = get_eq_args([
    '--n_iters', '50000',
    '--batch_size', '8',
    '--log_every', '500',
    '--ckpt_every', '5000',
    '--output_dir', OUTPUT_DIR,
    # w_equilibrium=0.12 (corrected from attempt 1's 1.0) is this
    # script's own default -- not overridden here.
])
n_elem = (DEFAULT_RESOLUTION[0] - 1) * (DEFAULT_RESOLUTION[1] - 1) * (DEFAULT_RESOLUTION[2] - 1)
print(f'\nTraining at {DEFAULT_RESOLUTION} -> {n_elem} elements, {args.n_iters} iterations, '
      f'w_equilibrium={args.w_equilibrium}, n_virtual={args.n_virtual}, checkpoints -> {OUTPUT_DIR}')
print('Watch energy -- it should behave like the DEM baseline\'s own training (settling '
      'toward a low value, not diverging or oscillating wildly). If it does not, interrupt '
      'and report rather than letting the full 50,000 iterations run on a still-broken loss.')

torch.cuda.reset_peak_memory_stats(device)
t0 = time.time()
model = train(args, device)
elapsed_total = time.time() - t0

print(f'\n{"=" * 90}')
print(f'Training finished in {elapsed_total:.1f}s ({elapsed_total / args.n_iters:.4f}s/iteration average).')
print(f'GPU peak memory: {torch.cuda.max_memory_allocated(device) / 1e6:.1f}MB allocated.')

print(f'\n{"=" * 90}')
print('=== Evaluating the final checkpoint (original evaluate_B3_qois.py -- u-only model) ===')
CHECKPOINT = f'{OUTPUT_DIR}/checkpoint_{args.n_iters}.pt'
DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
OUT_JSON = f'{OUTPUT_DIR}/qois_{args.n_iters}.json'

sys.argv = [
    'evaluate_B3_qois.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--out_json', OUT_JSON,
]
from omar_pfem.evaluate_B3_qois import main as evaluate_main
evaluate_main()

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(OUTPUT_DIR, kind='b3_equilibrium_augmented_training',
                    args={'n_iters': args.n_iters, 'batch_size': args.batch_size,
                          'w_equilibrium': args.w_equilibrium, 'n_virtual': args.n_virtual},
                    started_at=_started,
                    results={'elapsed_total_s': elapsed_total,
                             's_per_iter': elapsed_total / args.n_iters},
                    outputs=[CHECKPOINT, OUT_JSON],
                    notes="Second real training attempt of the equilibrium-augmented B3 "
                          "formulation (Professor Rabczuk's items 2/3). Attempt 1 "
                          "(w_equilibrium=1.0) failed ~100x worse than the DEM baseline. "
                          "Root-caused via a CPU measurement: the raw weak-form residual "
                          "is 6.5-10x larger than the energy term at production scale, so "
                          "w=1.0 let it dominate the loss. w_equilibrium corrected to 0.12 "
                          "(the measured ratio). See train_B3_equilibrium_augmented.py's "
                          "own 'STATUS (updated 2026-10-22)' docstring entry for the full "
                          "diagnosis.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back.')
